# Prithvi-EO-2.0 Proof-of-Concept — IOI / SYARIMO mill

**Thesis: Catching Greenwashing from Space** — professor-approved add-on comparing a
geospatial foundation model (IBM/NASA **Prithvi-EO-2.0-300M**) against the existing
Hansen Global Forest Change pipeline, for one representative mill.

**Mill selected:** IOI / SYARIMO, Malaysia (lat 5.334037, lon 117.781334) — IOI's
single highest post-2020 loss mill (12,317.8 ha), 10 km buffer, matching this
project's standard convention (`BUFFER_M = 10000`, Hansen GFC v1.13, scale 30 m
for the ground-truth layer).

## One-time manual steps (do these, then Runtime > Run all)
1. **Runtime > Change runtime type > T4 GPU** (free tier). If T4 is unavailable /
   disconnects repeatedly, switch to Colab Pro and pick a GPU there — everything
   else in this notebook is unchanged.
2. **Runtime > Run all.**
3. You will see **two** browser permission prompts during the run — click through both:
   - A **Google Drive mount** prompt (cell 2) — the three exported GeoTIFFs
     (`sentinel2_before_2019.tif`, `sentinel2_after_2024.tif`,
     `hansen_groundtruth.tif`) live in `My Drive/thesis_prithvi_poc/`, dropped
     there by `scripts/export_prithvi_poc.py` run locally against the
     `thesis-greenwashing` Earth Engine project. Use the **same Google account**
     that account is tied to.
   - **Only if** those files aren't in Drive yet (exports can take 2-10 min and
     may still be finishing), the fallback cell authenticates Earth Engine
     directly in Colab and re-pulls the same imagery — that needs one more
     OAuth click.
4. Everything after that runs unattended. Total runtime: ~3-6 min on a T4
   (model download + tiled inference over a 20x20 km region at 10 m).

## What this notebook does
- Loads the two Sentinel-2 composites (before: 2019, after: 2023-06..2024-12)
  and the Hansen ground-truth mask, all clipped to the identical 10 km-buffer bbox.
- Runs the **frozen** Prithvi-EO-2.0-300M ViT encoder over 224x224 px tiles of
  each composite (separately, since we're not fine-tuning), producing one
  embedding vector per tile per period.
- Computes **embedding cosine distance** (before vs after) per tile as an
  unsupervised "change intensity" score — this is the standard way to get a
  change signal out of a frozen foundation-model encoder without a fine-tuned
  downstream head.
- Aggregates Hansen post-2020 loss to the same tile grid and compares: does
  Prithvi's change score run high where Hansen says loss happened?
- Produces a side-by-side figure (S2 before / S2 after / Hansen loss mask /
  Prithvi change heatmap) plus a simple agreement metric (tile-level
  precision/recall/IoU and Spearman correlation).

See `Results/prithvi_poc_readme.md` in the repo for what "success" vs "honest
attempt" means for this POC, and the limitations section at the bottom of this
notebook.

## Run order — 9 steps, top to bottom

Run every cell in order (`Runtime > Run all` is fine). Do not skip step 7 —
skipping it is what causes `NameError: name 'loss_fraction_grid' is not defined`
in step 8.

| Step | Cell does | You need to do |
|---|---|---|
| 1 | Install dependencies | **Restart session** after this one (numpy gets downgraded) |
| 2 | Mount Google Drive, look for the 3 exported GeoTIFFs | Click through the Drive permission prompt |
| 3 | Fallback: re-pull imagery from Earth Engine | Only runs if step 2 printed MISSING for a file. Otherwise it prints "nothing to do" and does nothing — safe to run either way |
| 4 | Load the GeoTIFFs into memory | — |
| 5 | Load the frozen Prithvi-EO-2.0-300M model | Downloads ~1.3GB from Hugging Face, takes a minute |
| 6 | Tiled inference — builds `change_grid` | Needs GPU (T4), takes a few minutes |
| 7 | Aggregate Hansen ground truth to the same tile grid — builds `loss_fraction_grid` | **This is the step that was skipped before** |
| 8 | Agreement metrics — Spearman correlation, precision/recall/IoU | Needs `change_grid` (step 6) and `loss_fraction_grid` (step 7) — both must have run |
| 9 | Side-by-side comparison figure, saved to Drive | Final output |


In [2]:
# Cell 1 — install dependencies
# Note: Force numpy<2.0 to avoid 'ImportError: cannot import name _center'
!pip -q install "numpy<2.0" terratorch rasterio tifffile scipy scikit-learn matplotlib huggingface_hub
print("Dependencies installed. Please restart the session (Runtime > Restart session) if prompted or if import errors persist.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 105.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.9/44.9 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 560.3/560.3 kB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.9/161.9 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.6/35.6 MB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.0/244.0 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━

In [3]:
# Cell 2 — mount Drive and locate the exported GeoTIFFs
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_DIR = "/content/drive/MyDrive/thesis_prithvi_poc"
FILES = {
    "before": os.path.join(DRIVE_DIR, "sentinel2_before_2019.tif"),
    "after": os.path.join(DRIVE_DIR, "sentinel2_after_2024.tif"),
    "hansen": os.path.join(DRIVE_DIR, "hansen_groundtruth.tif"),
}

for name, path in FILES.items():
    print(name, "->", path, "FOUND" if os.path.exists(path) else "MISSING")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
before -> /content/drive/MyDrive/thesis_prithvi_poc/sentinel2_before_2019.tif FOUND
after -> /content/drive/MyDrive/thesis_prithvi_poc/sentinel2_after_2024.tif FOUND
hansen -> /content/drive/MyDrive/thesis_prithvi_poc/hansen_groundtruth.tif FOUND


In [4]:
# Cell 3 — FALLBACK ONLY: if any file above is MISSING (exports still running,
# or you're on a different Google account than the EE-linked Drive), this cell
# re-pulls the identical imagery directly from Earth Engine into Colab.
# Skip / do not run this cell if Cell 2 printed FOUND for all three files.

MISSING = [n for n, p in FILES.items() if not os.path.exists(p)]
print("Missing:", MISSING)

if MISSING:
    import ee
    ee.Authenticate()  # one-time OAuth click in Colab
    ee.Initialize(project='thesis-greenwashing')

    # Config mirrors data/mills/prithvi_poc_config.json (IOI / SYARIMO, 10km buffer)
    CONFIG = {
        "latitude": 5.334037, "longitude": 117.781334, "buffer_m": 10000,
        "s2_bands": ["B2", "B3", "B4", "B8", "B11", "B12"],
        "before_date_range": ["2019-01-01", "2019-12-31"],
        "after_date_range": ["2023-06-01", "2024-12-31"],
        "scale_s2_m": 10, "scale_hansen_m": 30,
    }
    point = ee.Geometry.Point([CONFIG["longitude"], CONFIG["latitude"]])
    region = point.buffer(CONFIG["buffer_m"]).bounds()

    def mask_clouds(img):
        scl = img.select("SCL")
        mask = scl.eq(4).Or(scl.eq(5)).Or(scl.eq(6)).Or(scl.eq(7))
        return img.updateMask(mask)

    def s2_composite(date_start, date_end):
        coll = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
                .filterDate(date_start, date_end).filterBounds(region)
                .map(mask_clouds).select(CONFIG["s2_bands"]))
        return coll.median().multiply(0.0001).clip(region)

    def download_geotiff(image, scale, out_path):
        img_safe = image.multiply(10000).toInt16()
        try:
            url = image.getDownloadURL({"region": region, "scale": scale, "format": "GEO_TIFF",
                                         "crs": "EPSG:4326", "maxPixels": 1e10})
        except Exception as e:
            if "must be less than or equal to" not in str(e):
                raise
            print(f"request too large at scale={scale}m, retrying at scale={scale*2}m with int16 cast")
            url = img_safe.getDownloadURL({"region": region, "scale": scale * 2, "format": "GEO_TIFF",
                                             "crs": "EPSG:4326", "maxPixels": 1e10})
        import urllib.request
        urllib.request.urlretrieve(url, out_path)
        print("downloaded ->", out_path)

    os.makedirs("/content/prithvi_fallback", exist_ok=True)
    if "before" in MISSING:
        FILES["before"] = "/content/prithvi_fallback/sentinel2_before_2019.tif"
        download_geotiff(s2_composite(*CONFIG["before_date_range"]), CONFIG["scale_s2_m"], FILES["before"])
    if "after" in MISSING:
        FILES["after"] = "/content/prithvi_fallback/sentinel2_after_2024.tif"
        download_geotiff(s2_composite(*CONFIG["after_date_range"]), CONFIG["scale_s2_m"], FILES["after"])
    if "hansen" in MISSING:
        gfc = ee.Image("UMD/hansen/global_forest_change_2025_v1_13")
        hansen_ref = (gfc.select("treecover2000").gt(30).rename("forest2000")
                      .addBands(gfc.select("lossyear").gte(21).rename("loss_post2020")).clip(region))
        FILES["hansen"] = "/content/prithvi_fallback/hansen_groundtruth.tif"
        download_geotiff(hansen_ref, CONFIG["scale_hansen_m"], FILES["hansen"])
else:
    print("All files found in Drive — nothing to do, continue to the next cell.")

Missing: []
All files found in Drive — nothing to do, continue to the next cell.


In [5]:
# Cell 4 — load the GeoTIFFs
import numpy as np
import rasterio

def read_tif(path):
    with rasterio.open(path) as src:
        arr = src.read()  # (bands, H, W)
        profile = src.profile
    return arr, profile

before_arr, before_profile = read_tif(FILES["before"])   # (6, H, W) reflectance 0-1
after_arr, after_profile = read_tif(FILES["after"])       # (6, H, W)
hansen_arr, hansen_profile = read_tif(FILES["hansen"])    # (2, Hh, Wh): forest2000, loss_post2020

before_arr = np.nan_to_num(before_arr, nan=0.0)
after_arr = np.nan_to_num(after_arr, nan=0.0)
before_arr = np.clip(before_arr, 0, 1)
after_arr = np.clip(after_arr, 0, 1)

print("before:", before_arr.shape, "after:", after_arr.shape, "hansen:", hansen_arr.shape)

# Crop before/after to the same (smaller) H, W so tiling lines up exactly.
H = min(before_arr.shape[1], after_arr.shape[1])
W = min(before_arr.shape[2], after_arr.shape[2])
before_arr = before_arr[:, :H, :W]
after_arr = after_arr[:, :H, :W]
print("cropped to common size:", H, "x", W)

before: (6, 2004, 2000) after: (6, 2004, 2000) hansen: (2, 669, 667)
cropped to common size: 2004 x 2000


In [6]:
# Cell 5 — load the frozen Prithvi-EO-2.0-300M encoder\nimport torch\nfrom terratorch.registry import BACKBONE_REGISTRY\n\ndevice = \"cuda\" if torch.cuda.is_available() else \"cpu\"\nprint(\"device:\", device)\n\n# Registry key naming has varied across terratorch versions - try the known\n# candidates in order so this cell doesn't hard-fail on a naming mismatch.\nCANDIDATE_NAMES = [\n    \"prithvi_eo_v2_300\",\n    \"prithvi_eo_v2_300m\",\n    \"ibm-nasa-geospatial/Prithvi-EO-2.0-300M\",\n]\n\nmodel = None\nlast_err = None\nfor name in CANDIDATE_NAMES:\n    try:\n        model = BACKBONE_REGISTRY.build(name, pretrained=True)\n        print(f\"Loaded backbone via registry key: '{name}'\")\n        break\n    except Exception as e:\n        last_err = e\n        print(f\"  '{name}' failed: {e}\")\n\nif model is None:\n    print(\"\\nAll registry keys failed. Listing available backbones so you can pick the right one:\")\n    try:\n        print(list(BACKBONE_REGISTRY.keys()))\n    except Exception:\n        pass\n    raise last_err\n\nmodel = model.to(device).eval()\nprint(\"Prithvi-EO-2.0-300M loaded, frozen (no fine-tuning) — encoder used purely for embeddings.\")

In [8]:
# Cell 6 — tiled inference: one embedding per 224x224 tile per period, then
# cosine distance (before vs after) as the unsupervised change score.

import torch
import numpy as np
from terratorch.registry import BACKBONE_REGISTRY

# Ensure device is defined
device = "cuda" if torch.cuda.is_available() else "cpu"

# Ensure model is defined/loaded
if 'model' not in globals():
    CANDIDATE_NAMES = ["prithvi_eo_v2_300", "prithvi_eo_v2_300m", "ibm-nasa-geospatial/Prithvi-EO-2.0-300M"]
    model = None
    for name in CANDIDATE_NAMES:
        try:
            model = BACKBONE_REGISTRY.build(name, pretrained=True)
            break
        except:
            continue
    if model is not None:
        model = model.to(device).eval()
    else:
        raise RuntimeError("Could not load Prithvi model backbone. Please check Cell 5.")

TILE = 224  # divisible by ViT patch size 16

def get_embedding(tile_chw):
    # tile_chw: (6, TILE, TILE) normalized reflectance -> (1, 6, 1, TILE, TILE)
    x = torch.from_numpy(tile_chw).float().unsqueeze(0).unsqueeze(2).to(device)
    with torch.no_grad():
        out = model(x)
    feat = out[-1] if isinstance(out, (list, tuple)) else out
    if feat.dim() == 3:       # (B, N_tokens, C) -> mean pool tokens
        feat = feat.mean(dim=1)
    elif feat.dim() > 2:      # (B, C, ...) spatial -> global average pool
        feat = feat.flatten(2).mean(dim=2)
    return feat.squeeze(0).float().cpu().numpy()

n_tiles_y = H // TILE
n_tiles_x = W // TILE
print(f"tile grid: {n_tiles_y} x {n_tiles_x} = {n_tiles_y * n_tiles_x} tiles of {TILE}px")

change_grid = np.zeros((n_tiles_y, n_tiles_x), dtype=np.float32)

for ty in range(n_tiles_y):
    for tx in range(n_tiles_x):
        y0, x0 = ty * TILE, tx * TILE
        b_tile = before_arr[:, y0:y0 + TILE, x0:x0 + TILE]
        a_tile = after_arr[:, y0:y0 + TILE, x0:x0 + TILE]
        combined = np.concatenate([b_tile, a_tile], axis=0).reshape(2, 6, TILE, TILE)
        mean = combined.mean(axis=(0, 2, 3), keepdims=True).squeeze(0)
        std = combined.std(axis=(0, 2, 3), keepdims=True).squeeze(0) + 1e-6
        b_norm = (b_tile - mean) / std
        a_norm = (a_tile - mean) / std

        emb_b = get_embedding(b_norm.astype(np.float32))
        emb_a = get_embedding(a_norm.astype(np.float32))
        cos_sim = np.dot(emb_b, emb_a) / (np.linalg.norm(emb_b) * np.linalg.norm(emb_a) + 1e-8)
        change_grid[ty, tx] = 1 - cos_sim

print("change_grid range:", change_grid.min(), "-", change_grid.max())

Prithvi_EO_V2_300M.pt: reconstructing file:   0%|          |  0.00B / 1.33GB            

Prithvi_EO_V2_300M.pt: downloading bytes:           |  0.00B            

tile grid: 8 x 8 = 64 tiles of 224px
change_grid range: 0.020126283 - 0.372315


In [9]:
# Cell 7 — aggregate Hansen ground truth to the same tile grid\nforest2000_full = hansen_arr[0]\nloss_full = hansen_arr[1]\n\n# Resample Hansen (30m) onto the S2 pixel grid (10m) via nearest-neighbor repeat\n# (30/10 = 3x), then crop/pad to the same H, W used for the tile grid above.\nscale_factor = 3\nhansen_loss_hi = np.kron(loss_full, np.ones((scale_factor, scale_factor)))[:H, :W]\nhansen_forest_hi = np.kron(forest2000_full, np.ones((scale_factor, scale_factor)))[:H, :W]\n\n# pad if kron output is smaller than H, W (edge rounding)\nif hansen_loss_hi.shape[0] < H or hansen_loss_hi.shape[1] < W:\n    pad_h = H - hansen_loss_hi.shape[0]\n    pad_w = W - hansen_loss_hi.shape[1]\n    hansen_loss_hi = np.pad(hansen_loss_hi, ((0, pad_h), (0, pad_w)), mode=\"edge\")\n    hansen_forest_hi = np.pad(hansen_forest_hi, ((0, pad_h), (0, pad_w)), mode=\"edge\")\n\nloss_fraction_grid = np.zeros((n_tiles_y, n_tiles_x), dtype=np.float32)\nfor ty in range(n_tiles_y):\n    for tx in range(n_tiles_x):\n        y0, x0 = ty * TILE, tx * TILE\n        loss_fraction_grid[ty, tx] = hansen_loss_hi[y0:y0 + TILE, x0:x0 + TILE].mean()\n\nprint(\"Hansen post-2020 loss fraction per tile — range:\", loss_fraction_grid.min(), \"-\", loss_fraction_grid.max())

In [ ]:
# Cell 8 — agreement metrics: does Prithvi's unsupervised change score track
# Hansen's labeled post-2020 loss, at tile level?
from scipy.stats import spearmanr
from sklearn.metrics import precision_score, recall_score, jaccard_score

# Self-heal: if Cell 7 was skipped, build loss_fraction_grid right here instead
# of failing with NameError. Needs hansen_arr, H, W, TILE, n_tiles_y, n_tiles_x
# from earlier cells — if those are also missing, re-run from Cell 4 onward.
if 'loss_fraction_grid' not in globals():
    print("loss_fraction_grid missing — Cell 7 was skipped, computing it now.")
    forest2000_full = hansen_arr[0]
    loss_full = hansen_arr[1]
    scale_factor = 3
    hansen_loss_hi = np.kron(loss_full, np.ones((scale_factor, scale_factor)))[:H, :W]
    hansen_forest_hi = np.kron(forest2000_full, np.ones((scale_factor, scale_factor)))[:H, :W]
    if hansen_loss_hi.shape[0] < H or hansen_loss_hi.shape[1] < W:
        pad_h = H - hansen_loss_hi.shape[0]
        pad_w = W - hansen_loss_hi.shape[1]
        hansen_loss_hi = np.pad(hansen_loss_hi, ((0, pad_h), (0, pad_w)), mode="edge")
        hansen_forest_hi = np.pad(hansen_forest_hi, ((0, pad_h), (0, pad_w)), mode="edge")
    loss_fraction_grid = np.zeros((n_tiles_y, n_tiles_x), dtype=np.float32)
    for ty in range(n_tiles_y):
        for tx in range(n_tiles_x):
            y0, x0 = ty * TILE, tx * TILE
            loss_fraction_grid[ty, tx] = hansen_loss_hi[y0:y0 + TILE, x0:x0 + TILE].mean()
    print("Hansen post-2020 loss fraction per tile — range:", loss_fraction_grid.min(), "-", loss_fraction_grid.max())

rho, pval = spearmanr(change_grid.ravel(), loss_fraction_grid.ravel())
print(f"Spearman correlation (Prithvi change score vs Hansen loss fraction): rho={rho:.3f}, p={pval:.4f}")

# Binarize: Hansen "lossy tile" = >30% of tile lost post-2020 (mirrors the
# treecover2000>30% canopy threshold convention used elsewhere in this repo).
hansen_positive = (loss_fraction_grid > 0.30).astype(int)

# Threshold Prithvi's change score at the same overall positive rate as Hansen
# (percentile matching) so the comparison isn't biased by an arbitrary cutoff.
target_rate = hansen_positive.mean()
if 0 < target_rate < 1:
    thresh = np.quantile(change_grid, 1 - target_rate)
    prithvi_positive = (change_grid >= thresh).astype(int)
    precision = precision_score(hansen_positive.ravel(), prithvi_positive.ravel(), zero_division=0)
    recall = recall_score(hansen_positive.ravel(), prithvi_positive.ravel(), zero_division=0)
    iou = jaccard_score(hansen_positive.ravel(), prithvi_positive.ravel(), zero_division=0)
    print(f"Tile-level agreement (top-{target_rate:.0%} of tiles by change score vs Hansen >30% loss tiles):")
    print(f"  precision={precision:.2f}  recall={recall:.2f}  IoU={iou:.2f}")
else:
    print("Hansen shows 0% or 100% lossy tiles in this region — precision/recall not meaningful, report Spearman rho only.")

In [ ]:
# Cell 9 — side-by-side comparison figure
import matplotlib.pyplot as plt

def s2_rgb(arr):
    # bands order B2,B3,B4,B8,B11,B12 -> RGB = B4,B3,B2 (indices 2,1,0)
    rgb = np.stack([arr[2], arr[1], arr[0]], axis=-1)
    p2, p98 = np.percentile(rgb, (2, 98))
    rgb = np.clip((rgb - p2) / (p98 - p2 + 1e-6), 0, 1)
    return rgb

fig, axes = plt.subplots(1, 4, figsize=(22, 6))

axes[0].imshow(s2_rgb(before_arr))
axes[0].set_title("Sentinel-2 BEFORE (2019)")

axes[1].imshow(s2_rgb(after_arr))
axes[1].set_title("Sentinel-2 AFTER (2023-2024)")

axes[2].imshow(hansen_loss_hi, cmap="Reds", vmin=0, vmax=1)
axes[2].set_title("Hansen post-2020 loss (ground truth)")

im3 = axes[3].imshow(change_grid, cmap="inferno")
axes[3].set_title("Prithvi-EO-2.0 change score (per 224px tile)")
plt.colorbar(im3, ax=axes[3], fraction=0.046)

for ax in axes:
    ax.axis("off")

fig.suptitle("IOI / SYARIMO, Malaysia — Prithvi-EO-2.0-300M vs Hansen GFC v1.13, 10km buffer", fontsize=14)
plt.tight_layout()

out_path = "/content/drive/MyDrive/thesis_prithvi_poc/prithvi_vs_hansen_comparison.png"
os.makedirs(os.path.dirname(out_path), exist_ok=True)
plt.savefig(out_path, dpi=200, bbox_inches="tight")
plt.show()
print("Saved comparison figure to Drive:", out_path)

## Interpretation notes and honest limitations

**What a positive result looks like:** Spearman rho meaningfully > 0 (e.g. > 0.3)
between Prithvi's unsupervised change score and Hansen's loss fraction, and/or
tile-level IoU noticeably above what random tile selection at the same positive
rate would give. That would support the claim that a frozen foundation-model
encoder picks up *some* deforestation signal without any deforestation-specific
training — useful corroborating evidence, not a replacement for Hansen.

**Known simplifications in this POC (report these plainly in the thesis, don't
hide them):**
1. **Frozen encoder, no fine-tuned head.** Prithvi-EO-2.0-300M was not
   fine-tuned for deforestation/change detection here — the change score is an
   embedding-distance proxy, not a calibrated probability. This is a legitimate,
   commonly used way to probe a foundation model's representations, but it's
   weaker evidence than a task-specific model.
2. **Tile-level (224 px / ~2.24 km at 10 m), not pixel-level.** Hansen operates
   at 30 m; this POC's comparison grid is coarser still. Don't claim pixel-level
   agreement.
3. **Per-tile self-normalization**, not verified official Prithvi pretraining
   statistics (those weren't confirmed from a trusted source at notebook-writing
   time). Document this as a chosen simplification.
4. **After-period composite built from only ~6 low-cloud Sentinel-2 scenes**
   (2023-06 to 2024-12) for this specific mill/region — coverage gaps are
   possible; check the AFTER panel in the figure for obvious holes before
   trusting a tile.
5. **One mill, one region.** This is a proof-of-concept for the thesis
   appendix, not a claim that Prithvi replaces or beats the Hansen pipeline
   across all 290 mills.

**If this doesn't work (honest-attempt fallback, per the 3-day hard-stop rule):**
If `terratorch`/model loading fails, or Colab disconnects repeatedly, or the
correlation is near zero / noisy and doesn't improve after checking the obvious
things (band order, reflectance scaling 0-1, tile alignment) — stop after 3 days
of attempts. Report in the thesis: "A Prithvi-EO-2.0 proof-of-concept was
attempted using the frozen 300M encoder on Sentinel-2 imagery for [mill]; exact
setup and failure mode documented in `Results/prithvi_poc_readme.md`." That is a
legitimate, defensible outcome for an appendix POC — the Hansen-based analysis
is the thesis's primary, load-bearing methodology regardless of how this goes.